In [ ]:
import pandas as pd
import os
from dataframe_modify.add_positions import add_positions
from dataframe_modify.swap_perspective import swap_y
from dataframe_modify.swap_perspective import generalize_perspective_sb360
from dataframe_modify.grid_areas import add_grid_areas
from sequences.get_sequences import get_sequences, get_corners_sequences, get_sequences_by_possession
from sequences.find_fast_actions import find_fast_actions_forward_perspective
from fast_actions.get_dataframe import get_fast_action_row
from corners.metric import evaluate_corner
from corners.get_dataframe import get_df_row, prepare_matches_from_pkl
from tqdm.notebook import tqdm_notebook
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# input_name = ['esa_22_23_matches.pkl']
# output_name = ['corners_xgb_22_23_0613_2.pkl']
input_name = ['esa_22_23_matches.pkl', 'esa_21_22_matches.pkl','euro_matches.pkl', 'fifa_matches_2022.pkl']
output_name = ['corners_xgb_esa_22_23_0614.pkl','corners_xgb_esa_21_22_0614.pkl','corners_xgb_euro_0614.pkl', 'corners_xgb_fifa_2022_06114.pkl']

In [ ]:
params = {
    'pass_to_zone_I': -0.1,
    'pass_to_zone_II': 0.2,
    'pass_to_zone_III': 0.3,
    'pass_incomplete': -0.3,
    'pass_shot_assist': 0.5,
    'pass_goal_assist': 0.8,
    'pass_cross': 0.2,
    'pass_by_opponent': -0.2,
    'shot_goal': 3,
    'shot_saved': 2,
    'shot_blocked': 0.5,
    'shot_off_target': 0.8,
    'shot_post': 1,
    'shot_saved_to_post': 1,
    'shot_saved_off_target': 0.5,
    'clearance_by_opponent': -0.3,
    'ball_recovery_by_opponent': -0.3,
    'ball_recovery_by_own_team': 0.5,
    'foul_committed_by_own_team': -0.5,
    'foul_committed_by_own_team_with_card': -1,
    'foul_committed_by_opponent': 0.5,
    'foul_committed_by_opponent_with_penalty': 1,
}

In [ ]:
def prepare_corners_pkls(input_pkls_paths, output_pkls_paths, params):
    for input_path, output_path in zip(input_pkls_paths, output_pkls_paths):
        if input_path.startswith('euro') or input_path.startswith('fifa'):
            obv = False
        else:
            obv = True
        print('='*50)
        print(f'Processing {input_path}...')
        matches = prepare_matches_from_pkl(f'data/{input_path}')
        print('Prepairing sequences...')
        sequences = get_sequences_by_possession(matches)
        corners = get_corners_sequences(sequences)
        print(f'Number of corners: {len(corners)}')
        print('Evaluating corners...')
        corners = [evaluate_corner(corner, params) for corner in tqdm_notebook(corners, dynamic_ncols=True)]
        num_of_corners_failed = 0
        
        xgb_df = pd.DataFrame()
        # xgb_df = xgb_df.append([get_df_row(corner, end_pass_zones=True, shot_on_goal=True) for corner in corners])
        print('Creating dataframe...')
        for i, corner in tqdm_notebook(enumerate(corners), dynamic_ncols=True):
            try:
                xgb_df = xgb_df.append([get_df_row(corner, end_pass_zones=True, shot_on_goal=True, obv=obv)])
            except Exception as e:
                print(f'Corner {i} failed')
                num_of_corners_failed += 1
                print(e)
                continue
        print(f'Number of corners failed: {num_of_corners_failed}')
        xgb_df.to_pickle(f'data/{output_path}')
        print(f'Saved to {output_path}')         
        


In [ ]:
prepare_corners_pkls(input_name, output_name, params)

In [ ]:
def df_print_corner(df):
    corner_df = df[['period', 'timestamp', 'type', 'team', 'player', 'event', 'corner_score']]
    return corner_df